Ya hay muchos valores medidos:
1. Cantidad de operadores lógicos
2. Vocab size
3. Avergae words and average literals
4. Variedad temática

Hay que medir cosas nuevas y GOD:
1. Aridad en los predicados. -> Cantidad de predicados por aridad. (ESTO SE DEBE COMPARAR CON LA AUTOFORM DE LOS MODELOS)
2. Uso de constantes
3. Porcentajes (o mejor dicho distribuciones) de operadores lógicos.

In [1]:
import sys, re, spacy
sys.path.insert(0, '/home/flopezp/LogicSim')  
import pandas as pd
from logicsim import utils, metrics
from datasets import load_dataset
# La medición la vamos a hacer con el suite de LogicSim desarrollado por su servidor. VAMO LA PUTA MADRE.

## Datasets

In [2]:
val = r'/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_validation.jsonl'
train = r'/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_train.jsonl'
test = r'/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_test.jsonl'

def dataset_statistics2(dataset, ds_name, split=None):
    """
    Merge all splits of a dataset before running the existing analysis.
    The analysis below remains unchanged.
    """
    if ds_name == 'FOLIO':
        folio_splits = [
            pd.read_json(path, lines=True)
            for path in (train, val, test)
        ]
        dataset = pd.concat(folio_splits, ignore_index=True)['premises-FOL']

    elif ds_name == 'MALLS':
        malls_all = load_dataset('yuan-yang/MALLS-v0')
        dataset = (
            list(malls_all['train']['FOL'])
            + list(malls_all['test']['FOL'])
        )

    elif ds_name == 'Willow':
        willow_all = load_dataset('iedeveci/WillowNLtoFOL')
        dataset = (
            list(willow_all['train']['FOL_expression'])
            + list(willow_all['test']['FOL_expression'])
        )

    print('-'*45)
    print(f'Dataset: {ds_name}')
    print('Split: All')

    ds_arity = {1:0, 2:0, 3:0, 4:0, 5:0, 6:0}
    constants, logops = 0, 0

    for i in range(len(dataset)):
        elem1, elem2, elem3, _ = metrics.extract_info(dataset[i], True)

        if len(elem1) != 0:
            arity_count = metrics.get_arity_list(elem1)[1]
            for elem in arity_count:
                ds_arity[elem] += arity_count[elem]

        if len(elem2) != 0:
            constants += len(elem2)

        if len(elem3) != 0:
            logops += len(elem3)

    total_preds = sum(ds_arity.values())

    percentage = [
        round(ds_arity[elem] / total_preds, 5) * 100
        for elem in ds_arity
    ]

    for i in range(len(percentage)):
        print(f'Porcentaje de aridad {i+1}: {percentage[i]}')

dataset_statistics2(None, 'FOLIO')
dataset_statistics2(None, 'MALLS')
dataset_statistics2(None, 'Willow')

---------------------------------------------
Dataset: FOLIO
Split: All
Porcentaje de aridad 1: 57.025999999999996
Porcentaje de aridad 2: 39.877
Porcentaje de aridad 3: 2.924
Porcentaje de aridad 4: 0.172
Porcentaje de aridad 5: 0.0
Porcentaje de aridad 6: 0.0
---------------------------------------------
Dataset: MALLS
Split: All
Porcentaje de aridad 1: 89.78
Porcentaje de aridad 2: 8.532
Porcentaje de aridad 3: 1.4160000000000001
Porcentaje de aridad 4: 0.217
Porcentaje de aridad 5: 0.034
Porcentaje de aridad 6: 0.021
---------------------------------------------
Dataset: Willow
Split: All
Porcentaje de aridad 1: 87.701
Porcentaje de aridad 2: 11.985999999999999
Porcentaje de aridad 3: 0.214
Porcentaje de aridad 4: 0.099
Porcentaje de aridad 5: 0.0
Porcentaje de aridad 6: 0.0


Cosas para medir y tener más contexto:
1. Longitud predicados y constantes.
2. El análisis que viene en el paper de MALLS.
3. **hay que pensarle papus**

--------------
``multiword_elements()`` encuentra y junta los siguientes patrones lingüísticos:
1. ADJ + NOUN/PROPN + NOUN/PROPN
2. ADJ + NOUN/PROPN
3. NOUN/PROPN + NOUN/PROPN + NOUN/PROPN
4. NOUN/PROPN + NOUN/PROPN


Estaría bien cuantificar ese pedo.

In [3]:
# Funciones que DEBEN DE ENTRAR AL SUITE DE LOGICSIM
nlp = spacy.load('en_core_web_lg')

def multiword_elements(nl_texto):
    """
    A partir de una premisa en NL obtiene las sucesiones de adj-noun(-noun) o noun-noun(-noun).
    Se tiene que complementar con los valores individuales, pero con esto se evitan repeticiones.

    nl_texto = str ; La premisa NL a analizar.
    """
    process_text = re.sub(r'(  )+', '', nl_texto)
    doc = nlp(process_text)
    tagged = [[token.text, str(token.pos_)] for token in doc]
    composite_elements = []
    print(tagged)
    for i in range(len(tagged) - 1):
        if (tagged[i][1] == 'ADJ') and (tagged[i+1][1] in ['NOUN', 'PROPN']):
            try:
                assert i+2 <= len(tagged), 'Index out of bounds papu'
                if tagged[i+2][1] in ['NOUN', 'PROPN']:
                    texto = tagged[i][0]+tagged[i+1][0]+tagged[i+2][0]
                else: 
                    texto = tagged[i][0]+tagged[i+1][0]
                    
            except:
                continue

            if len(composite_elements) == 0:
                composite_elements.append(texto)
            elif not ((texto in composite_elements[-1]) and (texto != composite_elements[-1])):
                composite_elements.append(texto)

        if (tagged[i][1] in ['NOUN', 'PROPN']) and (tagged[i+1][1] in ['NOUN', 'PROPN']):
            try:
                assert i+2 <= len(tagged), 'Index out of bounds papu' # Lowkey siento que el assert es innecesario no?
                if tagged[i+2][1] in ['NOUN', 'PROPN']:
                    texto = tagged[i][0]+tagged[i+1][0]+ tagged[i+2][0]
                else:
                    texto = tagged[i][0] + tagged[i+1][0]
            except:
                continue
            
            if len(composite_elements) == 0:
                composite_elements.append(texto)
            elif not ((texto in composite_elements[-1]) and (texto != composite_elements[-1])):
                composite_elements.append(texto)

    if len(composite_elements) > 0:
        dict_ce = {elem: 0 for elem in composite_elements}
        return dict_ce
    else:
         return None
#    return composite_elements
# Hmmmm, no creo que sea necesario que se regrese un diccionario lowkirkiuenly. Vamos a ver kpdo con logicsim_back a ver si es necesario.
# Tenemos que regresarlo el dict. Pero se puede rehacer.

# ---------------------------------------------------------------------------------------------

def logicsim_back(texto, fol_texto, print_statements):
    """
    LogicSim+_b : Almost done. Need to modify arithmetic and noun-grouping.
    A nivel de código esto es asqueroso cabrón.
    
    Idea:
    LogOps(FOLIO) - LogOps(NL) = Una parte de LogicSim+Ret
    Lo idea es que sea un valor cercano a 0. Si es mayor que 0 entonces el modelo puede estar "optimizando" la
    traducción. Pero si el valor es negativo implica que está agregando cosas que no debería de estar diciendo.
    """
    ola = re.sub(r'(  )|\t|\"|\n', '', texto)
    print(ola)
    print(fol_texto[0])
    # --------- Contamos logops ---------
    forall = fol_texto[0].count('∀')
    land = fol_texto[0].count('∧')
    implies = fol_texto[0].count('→')
    xor = fol_texto[0].count('⊕')
    neg = fol_texto[0].count('¬')
    lor = fol_texto[0].count('∨')
    exists = fol_texto[0].count('∃')

    # --------- Contamos NL values ---------
    texto_low = ola.lower()
    every, aand, then, either, nott, orr, exist = 0, 0, 0, 0, 0, 0, 0
    every = texto_low.count('every ')
    aand = texto_low.count(' and ')
    then = texto_low.count(' then ')
    either = texto_low.count('either ')
    nott = texto_low.count(' not ')
    orr = texto_low.count(' or ')
    exist = texto_low.count('exists ')

    # --------- Sumamos y generamos diferencias --------- kpdo que si comentas así se ve como si fuera chatcito
    total_fol = forall + land + implies + xor + neg + lor + exists
    total_nl = every + aand + then + either + nott + orr + exist

    # --------- Preds/Consts vs NL ---------
    doc = nlp(texto)
    tagged = [[token.text, token.pos_] for token in doc]
    pred_const_nl = [elem[0].lower() for elem in tagged if elem[1] in ['PROPN', 'NOUN', 'ADJ', 'VERB']]
    multiword_pred_const = multiword_elements(texto)

    const_dict = metrics.get_nice_dict(r'(\([A-z0-9 ]+(\, [A-z0-9 ]{1,}){0,}\))', fol_texto, True)
    pred_dict = metrics.get_nice_dict(r'[A-z0-9]+\(([A-z0-9]+(,? [A-z0-9]+)*)\)', fol_texto, False)

    # ---------------------------------------------------------
    # Eliminand referencias dobles. Siento que hice DEMASIADAS VUELTAS CON ESTO IDK
    # ---------------------------------------------------------
    if multiword_pred_const != None:
        mw_keys = multiword_pred_const.keys() # Tiene que ser cuadrático chingo a mi madre.
        for value in pred_const_nl:
            index = pred_const_nl.index(value) # Índice del valor en cuestión.
            for element in mw_keys:
                if value in element:
                    pred_const_nl[index] = None
                    multiword_pred_const[element] += 1

        while None in pred_const_nl:
            pred_const_nl.remove(None)

        for value in multiword_pred_const:
            if multiword_pred_const[value] > 0:
                pred_const_nl.append(value)
    # ---------------------------------------------------------

    keys = list(pred_dict.keys())
    other_keys = list(const_dict.keys())
    
    fol_values = 0 #¿Qué vergas eres brother?
    for elem in keys:
        fol_values += pred_dict[elem]
    for elem in other_keys:
        fol_values += const_dict[elem]
        
    fol_values += total_fol # Esta es la cantidad de consts+pred+logops en FOL
    # -----------------------------------------------------------------------------
    nl_logops = ['every', 'and', 'then', 'either', 'not', 'or', 'exists']
    # Evitamos cuentas dobles de algunos cuantificadores.
    for elem in pred_const_nl:
        if elem in nl_logops:
            total_nl -= 1
    nl_values = len(pred_const_nl) + total_nl # Cantidad de consts+pred+logops en NL
    # -----------------------------------------------------------------------------

    logicsim_r = fol_values - nl_values

    if len(keys) > 0:
        for j in range(len(pred_dict)):
            split = keys[j].split('(')[0]
            if split in list(pred_dict.keys()):
                pred_dict[split] += pred_dict.pop(keys[j])
            else:
                pred_dict[split] = pred_dict.pop(keys[j])
        if print_statements:
            print('Predicados FOL: {}'.format(pred_dict))
    #else:
    #    print('No hay predicados')

    if print_statements:
    # --------- Print statements ---------
        #print("-"*5, 'Cuantificadores', '-'*5)
        #print('Constantes FOL: {}'.format(const_dict))
        #print('Consts y Preds PoST: {}'.format(pred_const_nl))
        #print("∀: {}. ∧: {}. →: {}. ⊕: {}. ¬: {}. ∨: {}. ∃: {}.".format(forall, land, implies, xor, neg, lor, exists)) 
        #print("Every: {}. And: {}. Then: {}. Either: {}. Not: {}. Or: {}. Exists: {}.".format(every, aand, then, either, nott, orr, exist)) 
        #print("Diferencias totales: {}".format(abs(logops_diffs))) # Mide algo distinto wtf
        print('Valores FOL: {}. Valores NL: {}.'.format(fol_values, nl_values))
        print('LogicSim+_b = {}'.format(logicsim_r))
        print('='*15)
    return logicsim_r


In [4]:
def count_lexical_patterns(nl_texto):
    """
        Regresa valores que pueden ser predicadoes o constantes. Hay 5 patrones léxicos que extrae:
            1. ADJ + NOUN/PROPN + NOUN/PROPN
            2. ADJ + NOUN/PROPN
            3. 3*NOUN/PROPN
            4. 2*NOUN/PROPN
            5. Single NOUN/PROPN
        Estos patrones solo se extraen de manera AISLADA. El patrón 5 solo se extrae cuando se tiene un NOUN/PROPN AISLADO
        También regresa los siguientes valores:
            6. Adjetivos
            7. Verbos
    """
    process_text = re.sub(r'(  )+', '', nl_texto)
    doc = nlp(process_text)
    tagged = [[token.lemma_, str(token.pos_)] for token in doc] #XD
    tags = [elem[1] for elem in tagged]
    #print(tags)
    # wow
    adj_n_n = []
    adj_n = []
    n_n_n = []
    n_n = []
    single_n = []
    adj = []
    verb = []
    composite_elements = []
    for i in range(len(tagged)):
        if i < len(tagged) - 1:
            if (tagged[i][1] == 'ADJ') and (tagged[i+1][1] in ['NOUN', 'PROPN']):
                try:
                    assert i+2 <= len(tagged), 'Index out of bounds papu'
                    if tagged[i+2][1] in ['NOUN', 'PROPN']:
                        texto = tagged[i][0]+tagged[i+1][0]+tagged[i+2][0]
                        adj_n_n.append(texto)
                        composite_elements.append(texto)
                        continue
                    else: 
                        texto = tagged[i][0]+tagged[i+1][0]
                        adj_n.append(texto)
                        composite_elements.append(texto)
                        continue
                        
                except:
                    continue

            if (tagged[i][1] in ['NOUN', 'PROPN']) and (tagged[i+1][1] in ['NOUN', 'PROPN']):
                try:
                    assert i+2 <= len(tagged), 'Index out of bounds papu' # Lowkey siento que el assert es innecesario no?
                    if tagged[i+2][1] in ['NOUN', 'PROPN']:
                        texto = tagged[i][0]+tagged[i+1][0]+ tagged[i+2][0]
                        n_n_n.append(texto)
                        composite_elements.append(texto)
                        continue
                    else:
                        texto = tagged[i][0] + tagged[i+1][0]
                        n_n.append(texto)
                        composite_elements.append(texto)
                        continue
                except:
                    continue

            if (tagged[i][1] in ['PROPN', 'NOUN']) and (tagged[i+1][1] not in ['NOUN', 'PROPN']) and (tagged[i-1][1] not in ['NOUN', 'PROPN']):
                single_n.append(tagged[i][0])
                continue

            if (tagged[i][1] in ['ADJ'] and tagged[i+1][1] not in ['NOUN', 'PROPN']):
                adj.append(tagged[i][0])
                continue

            if tagged[i][1] == 'VERB':
                verb.append(tagged[i][0])
                continue
        else:
            if tagged[i][1] == 'VERB':
                verb.append(tagged[i][0])
                continue
            if tagged[i][1] in ['PROPN', 'NOUN']:
                single_n.append(tagged[i][0])
                continue
            if tagged[i][1] == 'ADJ':
                adj.append(tagged[i][0])
                continue

    return composite_elements, adj_n_n, adj_n, n_n_n, n_n, single_n, adj, verb


ex1 = 'For all x and y, if x is a roof made of concrete and y is a roof made of seagrass, then x is stronger than y.'
ex2 = 'If a building has a roof made of concrete and another building has a roof made of seagrass, then the former is stronger than the latter.'
ex3 = 'The place visited by Max is a flight destination and the place visited by Max is from Tweed Airport.'

print(count_lexical_patterns(ex1))
print(count_lexical_patterns(ex2))
print(count_lexical_patterns(ex3))

([], [], [], [], [], ['x', 'y', 'roof', 'concrete', 'y', 'roof', 'seagrass', 'y.'], ['strong'], ['make', 'make'])
([], [], [], [], [], ['building', 'roof', 'concrete', 'building', 'roof', 'seagrass'], ['former', 'strong', 'latter'], ['have', 'make', 'have', 'make'])
(['flightdestination', 'TweedAirport'], [], [], [], ['flightdestination', 'TweedAirport'], ['place', 'Max', 'place', 'Max'], [], ['visit', 'visit'])


In [7]:
val = r'/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_validation.jsonl'
train = r'/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_train.jsonl'
test = r'/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_test.jsonl'

def counting_patterns(ds_name):
    if ds_name == 'FOLIO':
        folio_splits = [
            pd.read_json(path, lines=True)
            for path in (train, val, test)
        ]
        retranslation_list = pd.concat(folio_splits, ignore_index=True)['conclusion']
        fol_list = pd.concat(folio_splits, ignore_index=True)['conclusion-FOL']

    elif ds_name == 'MALLS':
        malls_all = load_dataset('yuan-yang/MALLS-v0')
        retranslation_list = list(malls_all['train']['NL']) + list(malls_all['test']['NL'])
        fol_list = list(malls_all['train']['FOL']) + list(malls_all['test']['FOL'])

    elif ds_name == 'Willow':
        willow_all = load_dataset('iedeveci/WillowNLtoFOL')
        retranslation_list = list(willow_all['train']['NL_sentence']) + list(willow_all['test']['NL_sentence'])
        fol_list = list(willow_all['train']['FOL_expression']) + list(willow_all['test']['FOL_expression'])

    print('-'*45)
    print(f'\t Analyzando {ds_name}...')
    print('-'*45)

    retranslation_list = list(retranslation_list)
    fol_list = list(fol_list)

    #print(retranslation_list[:5])
    #print(fol_list[:5])

    adj2n_count, adj1n_count, n3_count, n2_count, single_n, adj_count, verb_count = 0, 0, 0, 0, 0, 0, 0 
    predicates, constants = 0, 0
    constant_list = []

    for i in range(len(retranslation_list)):
        nl_elem = retranslation_list[i]
        fol_elem = fol_list[i]

        #print(nl_elem)
        #print(fol_elem) 
        
        pred_dict = metrics.get_nice_dict(r'[¬A-z0-9]+\(([A-z0-9]+(,? [A-z0-9]+)*)\)', [fol_elem], False) # Me lleva la putísima verga CON LAS PERRAS LISTAS CABRÓN.
        const_dict = metrics.get_nice_dict(r'(\([A-z0-9 ]+(\, [A-z0-9 -]{1,}){0,}\))', [fol_elem], True)
        _, adj2n, adj1n, n3, n2, n1, adj, vrb = count_lexical_patterns(nl_elem)
        pred_list = [elem.split('(')[0] for elem in list(pred_dict.keys())]
        pred_list = list(set([re.sub('¬', '', elem).lower() for elem in pred_list]))
        const_list = list(set([elem.lower() for elem in const_dict.keys()]))

        adj2n = list(set([elem.lower() for elem in adj2n]))
        adj1n = list(set([elem.lower() for elem in adj1n]))
        n3 = list(set([elem.lower() for elem in n3]))
        n2 = list(set([elem.lower() for elem in n2]))
        n1 = list(set([elem.lower() for elem in n1]))
        adj = list(set([elem.lower() for elem in adj]))
        vrb = list(set([elem.lower() for elem in vrb]))
        
        #print(f'Predicados LS: {pred_list}')
        #print(f'Constantes LS: {const_list}')

        # Esta suma no hace lo que queremos we.
        # Tal vez podemos hacer la suma hasta el final de los valores, pero es difícil rastrear el origen una vez que hagamos el set.
        adj2n_count += len(adj2n)
        adj1n_count += len(adj1n)
        n3_count += len(n3)
        n2_count += len(n2)
        single_n += len(n1)
        adj_count += len(adj)
        verb_count += len(vrb)

        predicates += len(pred_list)
        constants += len(const_list)

        current_list = []
        for elem in [adj2n, adj1n, n3, n2, n1, adj, vrb]:
            if len(elem) != 0:
                for inner_value in elem:
                    current_list.append(inner_value)

        current_list = list(set(current_list))
        aux = []
        for i in range(len(current_list)):
            current_len = len(aux)
            for j in range(i+1, len(current_list)):
                if current_list[i] in current_list[j]:
                    aux.append(current_list[j])
                    continue
                if current_list[j] in current_list[i]:
                    aux.append(current_list[i])
                    continue
            if (len(aux) == current_len) and (current_list[i] not in aux):
                aux.append(current_list[i])

        # NO ES TANTO CON CONJUNTOS, SI NO MÁS BIEN CON EMPAREJAMIENTOS DE SIMILITUD LÉXICA CAWN.
        logicsim_values = set(const_list).union(set(pred_list))
        aux_set = set(aux)
        ls_INTER_spacy = logicsim_values.intersection(aux_set)
        
        #print(f'Current_list values: {current_list}')
        #print(f'Spacy values: {list(set(aux))}')
        #print(f'Intersección LogicSim \cup SpacyExtraction: {ls_INTER_spacy}')
        #print('-'*25)

    total = adj2n_count + adj1n_count + n3_count + n2_count + single_n + adj_count + verb_count
    total_log  = predicates + constants

    print(' -- Contadores finales -- ')
    print(f'ADJ + 2(NOUN/PROPN): {adj2n_count}')
    print(f'ADJ + NOUN/PROPN: {adj1n_count}')
    print(f'3(NOUN/PROPN): {n3_count}')
    print(f'2(NOUN/PROPN): {n2_count}')
    print(f'1(NOUN/PROPN): {single_n}')
    print(f'VERBS: {verb_count}')
    print(f'ADJECTIVES: {adj_count}')
    print(f'Total PoST values Found: {total}')
    print('-'*15)
    print(' -- Valores Lógicos --')
    print(f'Predicados: {predicates}')
    print(f'Constantes: {constants}')
    print(f'Total LogValues found: {total_log}')
    print('-'*15)
    print(f'PoST/LogValues: {round((total/total_log)*100, 3)}')



counting_patterns('FOLIO')
counting_patterns('MALLS')
counting_patterns('Willow')

---------------------------------------------
	 Analyzando FOLIO...
---------------------------------------------
 -- Contadores finales -- 
ADJ + 2(NOUN/PROPN): 111
ADJ + NOUN/PROPN: 481
3(NOUN/PROPN): 207
2(NOUN/PROPN): 1030
1(NOUN/PROPN): 2908
VERBS: 1283
ADJECTIVES: 413
Total PoST values Found: 6433
---------------
 -- Valores Lógicos --
Predicados: 2581
Constantes: 2255
Total LogValues found: 4836
---------------
PoST/LogValues: 133.023
---------------------------------------------
	 Analyzando MALLS...
---------------------------------------------
 -- Contadores finales -- 
ADJ + 2(NOUN/PROPN): 3050
ADJ + NOUN/PROPN: 25251
3(NOUN/PROPN): 1003
2(NOUN/PROPN): 13495
1(NOUN/PROPN): 108246
VERBS: 58003
ADJECTIVES: 14941
Total PoST values Found: 223989
---------------
 -- Valores Lógicos --
Predicados: 124401
Constantes: 2034
Total LogValues found: 126435
---------------
PoST/LogValues: 177.157
---------------------------------------------
	 Analyzando Willow...
-----------------------

# EN VEZ DE HACER LO DE LA CELDA DE ABAJO PODEMOS COMPARAR LA EXTRACCIÓN DE SPACY VS LOGICSIM Y DETERMINAR SI TODO ELEMENTO DE NL TIENE UNA EXPRESIÓN CERCANA EN NL

$FOLinNL(foltext, nltext) =  $ Apariciones de folvalues en nltext

$NLinFOL(foltext, nltext) =  $ Apariciones de nltextvalues en foltext

In [34]:
# Tenemos que comparar estos valores con los predicados/constantes finales.

const_count, pred_count = 0, 0

conc_fol = folio['conclusion-FOL'].to_list()
const_list = []
pred_list = []
for elem in conc_fol:
    pred_dict = metrics.get_nice_dict(r'[A-z0-9¬]+\(([A-z0-9]+(,? [A-z0-9]+)*)\)', [elem], False) # Me lleva la putísima verga CON LAS PERRAS LISTAS CABRÓN.
    const_dict = metrics.get_nice_dict(r'(\([A-z0-9 ]+(\, [A-z0-9 -]{1,}){0,}\))', [elem], True)

    const_count += len(list(const_dict.keys()))
    pred_count += len(list(pred_dict.keys()))
    if len(const_dict) != 0:
        lista = list(const_dict.keys())
        for elem in lista:
            const_list.append(elem)
        del lista

    if len(pred_dict) != 0:
        lista = list(pred_dict.keys())
        for elem in lista:
            pred_list.append(elem)
        del lista


pred_list = [elem.split('(')[0] for elem in pred_list]

print(f'Total preds: {pred_count}')
print(f'Total const: {const_count}')
print(f'Porcentaje de constantes encontradas: {round(total/(const_count + pred_count), 4)*100}%')

Total preds: 408
Total const: 312
Porcentaje de constantes encontradas: 133.19%
